In [2]:
import pandas as pd

# 1. Load metadata
meta = pd.read_csv("/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata.csv", dtype=str)
meta["Plasmid ID"] = meta["Plasmid ID"].astype(str)

# 2. Expanded list of eukaryote-indicative keywords/phrases
euk_keywords = [
    # Host/organism names
    r"human",
    r"homo\s*sapiens",
    r"mouse",
    r"mus\s*musculus",
    r"rat",
    r"zebrafish",
    r"danio\s*rerio",
    r"drosophila",
    r"caenorhabditis",
    r"c\.?\s*elegans",
    r"arabidopsis",
    r"plant",
    r"saccharomyces",
    r"pichia",
    r"insect",
    r"sf9",
    r"cho",
    r"chinese\s*hamster\s*ovary",
    r"hek293",
    r"293t",
    r"hela",

    # Vector backbones / promoters / expression
    r"pcdna",
    r"pepsi",           # e.g. pEGFP‐N1 or related
    r"pegfp",
    r"pgk",
    r"pcmvgfp",
    r"pcmv",
    r"cmv",
    r"sv40",
    r"ef1a",
    r"tk\s*promoter",
    r"cag",             # chicken β-actin promoter often in euk vectors
    r"yeast\s*expression",
    r"yeast\s*shuttle",
    r"shuttle\s*vector",
    r"baculovirus",
    r"viral\s*vector",
    r"lentiviral",
    r"retroviral",
    r"aav",

    # Selectable markers in eukaryotes
    r"neo",
    r"G418",
    r"hygromycin",
    r"puromycin",
    r"blasticidin",
    r"ura3",
    r"leu2",
    r"his3",

]

# 3. Build a regex pattern, joining with '|'
#    We use \s* to allow optional whitespace between words (e.g. 'c\.?\s*elegans')
pattern = "|".join(euk_keywords)

# 4. Check each relevant column
backbone_flag = meta["Backbone"].str.contains(pattern, case=False, na=False, regex=True)
insert_flag   = meta["Gene/Insert"].str.contains(pattern, case=False, na=False, regex=True)
cloning_flag  = meta["Cloning Information"].str.contains(pattern, case=False, na=False, regex=True)

# 5. Combine
meta["contains_euk_meta"] = backbone_flag | insert_flag | cloning_flag

# 6. Summary
counts = meta["contains_euk_meta"].value_counts()
print("Metadata‐based eukaryotic flags:")
print(counts)
print(f"\nFraction flagged: {counts.get(True,0)/len(meta):.2%}")

# 7. Inspect some examples
print("\nExamples of plasmids flagged:")
print(
    meta.loc[meta["contains_euk_meta"], 
             ["Plasmid ID", "Backbone", "Gene/Insert", "Cloning Information"]]
        .head(10)
)


Metadata‐based eukaryotic flags:
contains_euk_meta
False    9231
True     1073
Name: count, dtype: int64

Fraction flagged: 10.41%

Examples of plasmids flagged:
   Plasmid ID                                           Backbone  \
1       87282  Vector backbone pRS414; Backbone manufacturer ...   
15      37551  Vector backbone pRS306; Backbone size w/o inse...   
22     106468  Vector backbone pKP112; Total vector size (bp)...   
26      87293  Vector backbone pGADT7; Backbone manufacturer ...   
29      40235  Vector backbone pRS413; Backbone size w/o inse...   
34     112894  Vector backbone pSLIK; Backbone manufacturer I...   
42      44564  Vector backbone pVV16, pSE100, pMC1m; Backbone...   
43     105167  Vector backbone pFA6a; Backbone size w/o inser...   
46      26262  Vector backbone pCaSpeR4; Backbone size w/o in...   
59      40868  Vector backbone pFastBac1; Vector type Bacteri...   

      Gene/Insert Cloning Information  
1   Not Available       Not Available  
15  Not A

In [3]:
# inspect columns 
print("\nColumns in metadata:")
print(meta.columns.tolist())



Columns in metadata:
['Unnamed: 0', 'Plasmid ID', 'Purpose', 'Depositing Lab', 'Publication', 'Backbone', 'Gene/Insert', 'Growth in Bacteria', 'Cloning Information', 'contains_euk_meta']


In [4]:
meta.head(5)

,Unnamed: 0,Plasmid ID,Purpose,Depositing Lab,Publication,Backbone,Gene/Insert,Growth in Bacteria,Cloning Information,contains_euk_meta
0,0,49693,Not Available,Depositing Lab Christopher Voigt,Rhodius et al Mol Syst Biol. 2013 Oct 29;9:702...,Vector backbone pVRa; Vector type Bacterial Ex...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available,False
1,1,87282,Purpose Encodes Hsh155 R294L H331D for express...,Depositing Lab Aaron Hoskins,Carrocci et al Nucleic Acids Res. 2017 Jan 6. ...,Vector backbone pRS414; Backbone manufacturer ...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available,True
2,2,91257,Purpose Protein expression and purification of...,Depositing Lab Sachdev Sidhu,Teyra et al Structure. 2017 Oct 3;25(10):1598-...,Vector backbone pHH0103; Vector type Bacterial...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available,False
3,3,91368,Purpose Protein expression and purification of...,Depositing Lab Sachdev Sidhu,Teyra et al Structure. 2017 Oct 3;25(10):1598-...,Vector backbone pHH0103; Vector type Bacterial...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available,False
4,4,11304,Not Available,Depositing Lab Sung-Hou Kim,Kim et al J Struct Biol. 1998 Jan . 121(1):76-80.,Vector backbone pET21a; Backbone manufacturer ...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available,False


In [5]:
# Find out escherichia coli plasmids
ecoli_keywords = [
    r"escherichia\s*coli",
    r"e\.?\s*coli",
    r"ecoli",
    r"bacterial\s",
    r"bacterium",
    r"bacteria"
]
# Build a regex pattern for E. coli
ecoli_pattern = "|".join(ecoli_keywords)
# Check the all column for E. coli
ecoli_flag = meta["Backbone"].str.contains(ecoli_pattern, case=False, na=False, regex=True) | \
              meta["Gene/Insert"].str.contains(ecoli_pattern, case=False, na=False, regex=True) | \
              meta["Cloning Information"].str.contains(ecoli_pattern, case=False, na=False, regex=True) | \
              meta["Growth in Bacteria"].str.contains(ecoli_pattern, case=False, na=False, regex=True)
# Add the E. coli flag to the metadata
meta["contains_ecoli_meta"] = ecoli_flag
# Summary for E. coli
ecoli_counts = meta["contains_ecoli_meta"].value_counts()
print("\nMetadata‐based E. coli flags:")
print(ecoli_counts)
print(f"\nFraction flagged: {ecoli_counts.get(True,0)/len(meta):.2%}")
# Inspect some examples of E. coli plasmids

# inspect backbone with full text
print("\nExamples of plasmids with full text in Backbone:")
print(
    meta.loc[meta["Backbone"].str.len() > 100, 
             ["Plasmid ID", "Backbone"]]
        .head(10)
)


Metadata‐based E. coli flags:
contains_ecoli_meta
True    10304
Name: count, dtype: int64

Fraction flagged: 100.00%

Examples of plasmids with full text in Backbone:
   Plasmid ID                                           Backbone
1       87282  Vector backbone pRS414; Backbone manufacturer ...
4       11304  Vector backbone pET21a; Backbone manufacturer ...
6      112558  Vector backbone pOP2S; Vector type Bacterial E...
7      100573  Vector backbone pGEX4T3; Backbone manufacturer...
8       15460  Vector backbone pCL15; Backbone manufacturer l...
12      35364  Vector backbone pBb; Backbone size w/o insert ...
15      37551  Vector backbone pRS306; Backbone size w/o inse...
17     118511  Vector backbone pFru97; Vector type Bacterial ...
18      92224  Vector backbone pBEST; Backbone manufacturer P...
22     106468  Vector backbone pKP112; Total vector size (bp)...


In [15]:
# unique Growth in Bacteria
'''print("\nUnique values in 'Growth in Bacteria':")
print(meta["Growth in Bacteria"].unique())'''

# number of unique Growth in Bacteria
unique_growth_bacteria = meta["Growth in Bacteria"].nunique()
print(f"\nNumber of unique values in 'Growth in Bacteria': {unique_growth_bacteria}")

# in first word of Growth in Bacteria
meta["First Word Growth in Bacteria"] = meta["Growth in Bacteria"].str.split().str[2]
print("new")
print("\nFirst word in 'Growth in Bacteria':")
print(meta["First Word Growth in Bacteria"].unique())
print(f"\nNumber of unique first words in 'Growth in Bacteria': {meta['First Word Growth in Bacteria'].nunique()}")

# Check for plasmids with no Growth in Bacteria
no_growth_bacteria = meta[meta["Growth in Bacteria"].isna()]
print(f"\nNumber of plasmids with no 'Growth in Bacteria': {len(no_growth_bacteria)}")

# Counts of first words in 'Growth in Bacteria'
first_word_counts = meta["First Word Growth in Bacteria"].value_counts()
print("\nCounts of first words in 'Growth in Bacteria':")
print(first_word_counts)


Number of unique values in 'Growth in Bacteria': 766
new

First word in 'Growth in Bacteria':
['Ampicillin,' 'Kanamycin,' 'Chloramphenicol,' 'Spectinomycin,'
 'Chloramphenicol' 'Hygromycin,' 'Tetracycline,' 'Streptomycin,'
 'Apramycin' 'Spectinomycin' 'Ampicillin' 'Gentamicin,' 'Nourseothricin'
 'Apramycin,' 'Bleocin' 'Erythromycin,' 'Trimethoprim;' 'Triclosan;'
 'Kanamycin' 'Tellurite;' 'Streptomycin' 'Ampicillin+' 'Blasticidin,']

Number of unique first words in 'Growth in Bacteria': 23

Number of plasmids with no 'Growth in Bacteria': 0

Counts of first words in 'Growth in Bacteria':
First Word Growth in Bacteria
Ampicillin,         5861
Kanamycin,          2714
Chloramphenicol,     888
Spectinomycin,       266
Chloramphenicol      130
Tetracycline,        113
Gentamicin,           76
Streptomycin,         72
Ampicillin            57
Hygromycin,           43
Nourseothricin        18
Apramycin,            13
Spectinomycin         12
Kanamycin              8
Bleocin                7


In [16]:
# speperate addgene plasmid fasta files into individual files
import os
import shutil
def separate_fasta_files(metadata, fasta_dir, output_dir):
    """
    Separate Addgene plasmid FASTA files into individual files based on metadata.
    
    Parameters:
    - metadata: DataFrame containing plasmid metadata with 'Plasmid ID' column.
    - fasta_dir: Directory containing the original FASTA files.
    - output_dir: Directory to save the separated FASTA files.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for _, row in metadata.iterrows():
        plasmid_id = row["Plasmid ID"]
        fasta_file = os.path.join(fasta_dir, f"{plasmid_id}.fasta")
        
        if os.path.exists(fasta_file):
            shutil.copy(fasta_file, os.path.join(output_dir, f"{plasmid_id}.fasta"))
        else:
            print(f"Warning: {fasta_file} does not exist.")

# Example usage
fasta_directory = "/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_fasta"
output_directory = "/cs/student/projects1/aibh/2024/acunning/Projects/Data/separated_fasta_files"
separate_fasta_files(meta, fasta_directory, output_directory)


In [2]:
from Bio import SeqIO
from pathlib import Path

# 1) Input multi-FASTA and output directory
input_fasta = "/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_sequences.fasta"
output_dir = Path("/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta")

# 2) Create the output directory if it doesn’t exist
output_dir.mkdir(parents=True, exist_ok=True)

# 3) Split into individual FASTA files with ADDGENE_ prefix,
#    but don’t overwrite existing files
count = 0
for record in SeqIO.parse(str(input_fasta), "fasta"):
    outfile = output_dir / f"ADDGENE_{record.id}.fasta"
    if not outfile.exists():
        SeqIO.write(record, str(outfile), "fasta")
        count += 1

print(f"Added {count} new ADDGENE FASTA files to {output_dir}")




Added 10304 new ADDGENE FASTA files to /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta


In [5]:
# inspect fasta files in /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta 

# inspect fasta files in /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta
import os       
fasta_dir = "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta"
fasta_files = [f for f in os.listdir(fasta_dir) if f.endswith('.fasta')]
print(f"Number of FASTA files in {fasta_dir}: {len(fasta_files)}")
print("First 10 FASTA files:")
for fasta_file in fasta_files[:10]:
    print(fasta_file)
print(f"\nTotal FASTA files: {len(fasta_files)}")   

# and total .fa files
fa_files = [f for f in os.listdir(fasta_dir) if f.endswith('.fa')]
print(f"Number of .fa files in {fasta_dir}: {len(fa_files)}")
print("First 10 .fa files:")
for fa_file in fa_files[:10]:   
    print(fa_file)          
# Total .fa files
print(f"\nTotal .fa files: {len(fa_files)}")

# Total fa and fasta files
total_files = len(fasta_files) + len(fa_files)
print(f"\nTotal FASTA and .fa files: {total_files}")

Number of FASTA files in /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta: 19027
First 10 FASTA files:
ADDGENE_51522.fasta
ADDGENE_139766.fasta
ADDGENE_62562.fasta
ADDGENE_180283.fasta
ADDGENE_137079.fasta
ADDGENE_112739.fasta
ADDGENE_109809.fasta
ADDGENE_78286.fasta
ADDGENE_98242.fasta
ADDGENE_104289.fasta

Total FASTA files: 19027
Number of .fa files in /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta: 16123
First 10 .fa files:
GenBank_RRJM01000135.1.fa
GenBank_RZFX01000257.1.fa
GenBank_CP088053.1.fa
GenBank_CP050033.1.fa
GenBank_MH847582.1.fa
GenBank_GQ385326.1.fa
GenBank_CP103467.1.fa
GenBank_RRNM01000146.1.fa
GenBank_RRKN01000138.1.fa
GenBank_CP034385.1.fa

Total .fa files: 16123

Total FASTA and .fa files: 35150


In [ ]:
import pandas as pd

# 1. Load metadata
meta = pd.read_csv("/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata.csv", dtype=str)
meta.head(5)

,Unnamed: 0,Plasmid ID,Purpose,Depositing Lab,Publication,Backbone,Gene/Insert,Growth in Bacteria,Cloning Information
0,0,49693,Not Available,Depositing Lab Christopher Voigt,Rhodius et al Mol Syst Biol. 2013 Oct 29;9:702...,Vector backbone pVRa; Vector type Bacterial Ex...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available
1,1,87282,Purpose Encodes Hsh155 R294L H331D for express...,Depositing Lab Aaron Hoskins,Carrocci et al Nucleic Acids Res. 2017 Jan 6. ...,Vector backbone pRS414; Backbone manufacturer ...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available
2,2,91257,Purpose Protein expression and purification of...,Depositing Lab Sachdev Sidhu,Teyra et al Structure. 2017 Oct 3;25(10):1598-...,Vector backbone pHH0103; Vector type Bacterial...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available
3,3,91368,Purpose Protein expression and purification of...,Depositing Lab Sachdev Sidhu,Teyra et al Structure. 2017 Oct 3;25(10):1598-...,Vector backbone pHH0103; Vector type Bacterial...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available
4,4,11304,Not Available,Depositing Lab Sung-Hou Kim,Kim et al J Struct Biol. 1998 Jan . 121(1):76-80.,Vector backbone pET21a; Backbone manufacturer ...,Not Available,"Bacterial Resistance(s) Ampicillin, 100 μg/mL;...",Not Available


In [2]:
meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10304 entries, 0 to 10303
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           10304 non-null  object
 1   Plasmid ID           10304 non-null  object
 2   Purpose              10304 non-null  object
 3   Depositing Lab       10304 non-null  object
 4   Publication          10304 non-null  object
 5   Backbone             10304 non-null  object
 6   Gene/Insert          10304 non-null  object
 7   Growth in Bacteria   10304 non-null  object
 8   Cloning Information  10304 non-null  object
dtypes: object(9)
memory usage: 724.6+ KB


In [3]:
import pandas as pd
from pathlib import Path
from Bio import SeqIO

# ─── Paths ───────────────────────────────────────────
meta_csv = Path("/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata.csv")
fasta_dir = Path("/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta")
out_csv  = Path("/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata_w_stats.csv")

# ─── Load metadata ───────────────────────────────────
df = pd.read_csv(meta_csv)

# Drop the unwanted 'Unnamed: 0' column if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Prefix IDs with ADDGENE_
df["Plasmid ID"] = df["Plasmid ID"].apply(lambda x: f"ADDGENE_{x}")

# ─── Build a lookup for sequence stats ───────────────
def get_seq_stats(plasmid_id):
    fasta_path = fasta_dir / f"{plasmid_id}.fasta"
    if not fasta_path.exists():
        return None, None  # missing sequence
    
    record = next(SeqIO.parse(fasta_path, "fasta"))
    seq = str(record.seq)
    size = len(seq)
    gc = 100.0 * (seq.count("G") + seq.count("C")) / size if size > 0 else 0
    return size, gc

sizes, gcs = [], []

for pid in df["Plasmid ID"]:
    size, gc = get_seq_stats(pid)
    sizes.append(size)
    gcs.append(gc)

df["Size (bp)"] = sizes
df["GC"] = gcs

# ─── Save updated metadata ──────────────────────────
df.to_csv(out_csv, index=False)

print(f"✅ Updated CSV saved to {out_csv}")


✅ Updated CSV saved to /cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata_w_stats.csv


In [33]:
import pandas as pd

# 1. Load metadata
meta = pd.read_csv("/cs/student/projects1/aibh/2024/acunning/Projects/Data/metadata/ft_metadata/Ecoli_pdb_ft35.tsv", sep="\t")
meta.head()

,Plasmid_ID,Data_Source,Topology,Completeness,Size (bp),GC,Host,MOB_type(s),Predicted_Mobility,Primary_Cluster_ID,Secondary_Cluster_ID
0,DDBJ_AB017809.1,DDBJ,linear,incomplete,1619,0.355775,Escherichia coli,-,non-mobilizable,AE932,AO815
1,DDBJ_AB020531.1,DDBJ,linear,incomplete,6445,0.433049,Escherichia coli,-,non-mobilizable,AA176,AH860
2,DDBJ_AB023657.1,DDBJ,linear,incomplete,282,0.645390,Escherichia coli,-,non-mobilizable,AA664,AI937
3,"DDBJ_AB024946.1,IMGPR_plasmid_640048319_000001...","DDBJ,IMG-PR,RefSeq",linear,complete,68817,0.460250,"Escherichia coli,-",-,non-mobilizable,AB269,AJ980
4,DDBJ_AB038042.1,DDBJ,linear,incomplete,4070,0.615233,Escherichia coli,-,non-mobilizable,AE581,AO395


In [27]:
df = pd.read_csv("/cs/student/projects1/aibh/2024/acunning/Projects/Data/addgene_plasmid_metadata_w_stats.csv", sep="\t")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10304 entries, 0 to 10303
Data columns (total 1 columns):
 #   Column                                                                                                                  Non-Null Count  Dtype 
---  ------                                                                                                                  --------------  ----- 
 0   Plasmid ID,Purpose,Depositing Lab,Publication,Backbone,Gene/Insert,Growth in Bacteria,Cloning Information,Size (bp),GC  10304 non-null  object
dtypes: object(1)
memory usage: 80.6+ KB


In [22]:
from pathlib import Path
from collections import Counter

# Databases/prefixes to recognise (file names start with these)
DB_PREFIXES = [
    "GenBank", "RefSeq", "PLSDB", "IMGPR", "DDBJ",
    "COMPASS", "ENA", "Kraken2", "TPA", "ADDGENE"
]

# Folders to process, in order
FOLDERS = [
    "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta",
    "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/noncircular_complete_fasta",
    "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/incomplete_fasta",
    "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/30k_bp_fasta",
    "/cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/100k_bp_fasta",
]

# Only count these file suffixes (adjust if needed)
FASTA_SUFFIXES = {".fasta", ".fa"}  # add ".fa", ".fna", etc., if you use them

def detect_db(filename: str) -> str:
    for db in DB_PREFIXES:
        if filename.startswith(db + "_") or filename.startswith(db + "-") or filename.startswith(db):
            return db
    return "OTHER"

def count_fastas(folder: Path) -> Counter:
    c = Counter()
    for p in folder.iterdir():
        if p.is_file() and p.suffix in FASTA_SUFFIXES:
            db = detect_db(p.name)
            c[db] += 1
    return c

def print_counts(title: str, counts: Counter):
    total = sum(counts.values())
    print(f"\n=== {title} ===")
    for db in DB_PREFIXES + ["OTHER"]:
        print(f"{db:9s}: {counts.get(db, 0)}")
    print(f"{'TOTAL':9s}: {total}")

cumulative = Counter()

for folder_str in FOLDERS:
    folder = Path(folder_str)
    if not folder.exists():
        print(f"\n=== {folder} ===")
        print("Folder not found — skipping.")
        continue

    counts = count_fastas(folder)
    print_counts(str(folder), counts)

    # Update and show cumulative
    cumulative.update(counts)
    print_counts("CUMULATIVE", cumulative)



=== /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/fasta ===
GenBank  : 3022
RefSeq   : 245
PLSDB    : 211
IMGPR    : 1
DDBJ     : 368
COMPASS  : 334
ENA      : 0
Kraken2  : 0
TPA      : 0
ADDGENE  : 10304
OTHER    : 0
TOTAL    : 14485

=== CUMULATIVE ===
GenBank  : 3022
RefSeq   : 245
PLSDB    : 211
IMGPR    : 1
DDBJ     : 368
COMPASS  : 334
ENA      : 0
Kraken2  : 0
TPA      : 0
ADDGENE  : 10304
OTHER    : 0
TOTAL    : 14485

=== /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/noncircular_complete_fasta ===
GenBank  : 105
RefSeq   : 1
PLSDB    : 7
IMGPR    : 1593
DDBJ     : 2
COMPASS  : 10
ENA      : 0
Kraken2  : 0
TPA      : 0
ADDGENE  : 0
OTHER    : 0
TOTAL    : 1718

=== CUMULATIVE ===
GenBank  : 3127
RefSeq   : 246
PLSDB    : 218
IMGPR    : 1594
DDBJ     : 370
COMPASS  : 344
ENA      : 0
Kraken2  : 0
TPA      : 0
ADDGENE  : 10304
OTHER    : 0
TOTAL    : 16203

=== /cs/student/projects1/aibh/2024/acunning/Projects/Data/dataset/incomplete_fasta ===
G